1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [1]:
from datasets import load_dataset

# Load the train.csv using the dataset load_dataset api
dataset = load_dataset("csv", data_files="/content/train.csv")

Generating train split: 0 examples [00:00, ? examples/s]

In [2]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
        num_rows: 2000
    })
})


In [3]:
def preprocess(data):
  data["combined_text"] = data["prompt"] + " " + data["A"]
  return data

In [4]:
dataset["train"] = dataset["train"].map(preprocess)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
combinedText = dataset["train"][51]["combined_text"]
print(combinedText)
print("Length of the combined_text:", len(combinedText))

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Length of the combined_text: 614


2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [6]:
from transformers import AutoTokenizer
BERT_MODEL = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
# Tokenizer Details
print(tokenizer)
print(f"Total Vocabulary Size of the Tokenizer of bert-base-uncased: {tokenizer.vocab_size}")

BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
Total Vocabulary Size of the Tokenizer of bert-base-uncased: 30522


3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [8]:
print(tokenizer.SPECIAL_TOKENS_ATTRIBUTES)
sep_id = tokenizer.sep_token_id
print(f"SEP Token ID: {sep_id}")

['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token']
SEP Token ID: 102


4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [9]:
prompts = []
for row in dataset["train"]:
  prompts.append(row["prompt"])

In [10]:
input_tokens = tokenizer(prompts, padding="max_length", truncation=True, max_length=128,return_tensors="pt")

In [11]:
input_tokens["input_ids"].shape

torch.Size([2000, 128])

5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

In [20]:
from transformers import AutoModel
# Load the model
model = AutoModel.from_pretrained(BERT_MODEL)

# Tokenize the input
input = dataset["train"][0]["prompt"]
input_token = tokenizer(input, return_tensors="pt")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
print(input_token)

{'input_ids': tensor([[  101,  4060,  1996,  2190,  2825,  3437,  1024,  2054,  2003,  3235,
          2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,  7047,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])}


In [22]:
import torch
with torch.no_grad():
  output = model(**input_token)

In [28]:
print(f"Size of last_hidden_state: {output["last_hidden_state"].size()}")

Size of last_hidden_state: torch.Size([1, 31, 768])


7.  Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [36]:
print(f"Sum of first 5 float values in the [CLS] vector: {output["last_hidden_state"][0][0][:5].sum():.4f}")

Sum of first 5 float values in the [CLS] vector: -1.2001


8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [64]:
model_att = AutoModel.from_pretrained(BERT_MODEL, output_attentions=True)

input = "Light-ion fusion is a technique."
input_token = tokenizer(input, return_tensors="pt")

with torch.no_grad():
  output = model_att(**input_token)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [65]:
tokens = tokenizer.convert_ids_to_tokens(input_token["input_ids"][0])
print("Input Token:", input_token )
print("Ids:", tokens)

Input Token: {'input_ids': tensor([[  101,  2422,  1011, 10163, 10077,  2003,  1037,  6028,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Ids: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [66]:
fusion_idx = tokens.index('fusion')
cls_idx = 0
print(f"Index of [CLS]: {cls_idx}" )
print(f"Index of 'fusion': {fusion_idx}")

Index of [CLS]: 0
Index of 'fusion': 4


In [73]:
last_layer_attn = output.attentions[-1]
print(last_layer_attn.size())
# Batch 0 , Head 0, Query [CLS], Key 'fusion'
attention_weight = last_layer_attn[0,0,cls_idx,fusion_idx]
print(f"Attention Weight: {attention_weight:.4f}")

torch.Size([1, 12, 10, 10])
Attention Weight: 0.1025


9. Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [75]:
# Load the model
from sentence_transformers import SentenceTransformer, util
sent_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [76]:
prompt = dataset["train"][0]["prompt"]
option = dataset["train"][0]["B"]

# Convert to embeddings
prompt_embedding = sent_model.encode(prompt, convert_to_tensor= True)
option_embedding = sent_model.encode(option, convert_to_tensor= True)

In [79]:
cosine_score = util.cos_sim(prompt_embedding, option_embedding)
cosine_score

tensor([[0.7658]])

10. Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  